# Recognition vs. Generation on ARC-AGI-1This notebook is the full study run, end to end: data sampling, generation phase, distractor preparation, recognition phase, analysis, and chart generation.## SetupBefore running:- Clone the [ARC-AGI repository](https://github.com/fchollet/ARC-AGI) as a sibling folder to this project.- Install dependencies: `pip install -r requirements.txt`.- Add an Anthropic API key to a local `.env` file: `ANTHROPIC_API_KEY=sk-ant-...`.- Set a hard spend cap on your Anthropic account before running. Total API cost for the full study is approximately $15.The full reproducible study runs top to bottom from this notebook.

## Imports and configuration

In [ ]:
import json
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

from arc_1_recog_gen import (
    build_prompt,
    build_recognition_prompt,
    call_claude,
    parse_recognition_response,
    parse_response,
    perturb_grid,
    score_attempt,
)

In [ ]:
# Anchor paths to project root
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ARC_PATH = PROJECT_ROOT / "ARC-AGI" / "data" / "evaluation"

print(f"Project root: {PROJECT_ROOT}")
print(f"ARC path: {ARC_PATH}")
print(f"Path exists: {ARC_PATH.exists()}")

task_files = list(ARC_PATH.glob("*.json"))
print(f"Found {len(task_files)} task files")

## Item samplingSample 30 items from the ARC-AGI-1 public eval set, stratified by maximum grid dimension into three buckets: small (≤10), medium (11-20), and large (21-30). Sample 10 from each bucket using a fixed seed for reproducibility.

In [ ]:
def max_grid_dim(task):
    """Return the maximum grid dimension across all train and test grids in a task."""
    dimensions = []
    for pair in task["train"] + task["test"]:
        for grid in [pair["input"], pair["output"]]:
            dimensions.append(max(len(grid), len(grid[0])))
    return max(dimensions)


# Compute max dim for every task
all_task_paths = list(ARC_PATH.glob("*.json"))
task_dims = []
for path in all_task_paths:
    with open(path) as f:
        task = json.load(f)
    task_dims.append((path.stem, max_grid_dim(task)))

print(f"Computed dims for {len(task_dims)} tasks")

# Bucket by size
small = [tid for tid, dim in task_dims if dim <= 10]
medium = [tid for tid, dim in task_dims if 11 <= dim <= 20]
large = [tid for tid, dim in task_dims if 21 <= dim <= 30]
print(f"Small (≤10): {len(small)}, Medium (11-20): {len(medium)}, Large (21-30): {len(large)}")

In [ ]:
# Sample 10 from each bucket with fixed seed
random.seed(42)

sampled_task_ids = []
for bucket in [small, medium, large]:
    sampled_task_ids.extend(random.sample(bucket, 10))

print(f"Sampled {len(sampled_task_ids)} task IDs")

# Save the sample as a fixed artifact
sample_path = PROJECT_ROOT / "data" / "sampled_tasks.json"
sample_path.parent.mkdir(exist_ok=True)
with open(sample_path, "w") as f:
    json.dump(sampled_task_ids, f, indent=2)
print(f"Saved sample to {sample_path}")

## Generation phaseFor each of the 30 sampled tasks, run k=3 generation attempts with extended thinking enabled (8000-token budget). Score each attempt with binary correctness and cell-level Hamming similarity to ground truth. Save results to `data/generation_results.json`.This phase makes 90 API calls and takes roughly 30-60 minutes. Estimated cost: ~$10.

In [ ]:
NUM_ATTEMPTS = 3
RESULTS_FILE = PROJECT_ROOT / "data" / "generation_results.json"

with open(PROJECT_ROOT / "data" / "sampled_tasks.json") as f:
    sampled_ids = json.load(f)

all_results = []

for task_index, task_id in enumerate(sampled_ids):
    task_path = ARC_PATH / f"{task_id}.json"
    with open(task_path) as f:
        task = json.load(f)

    prompt = build_prompt(task)
    ground_truth = task["test"][0]["output"]

    for attempt_index in range(NUM_ATTEMPTS):
        api_response = call_claude(prompt, thinking_budget=8000)
        parsed = parse_response(api_response["text"])
        score = score_attempt(parsed, ground_truth)

        result = {
            "task_id": task_id,
            "attempt": attempt_index,
            "model": api_response["model"],
            "timestamp": api_response["timestamp"],
            "input_tokens": api_response["input_tokens"],
            "output_tokens": api_response["output_tokens"],
            "response_text": api_response["text"],
            "parsed_answer": parsed,
            "ground_truth": ground_truth,
            "status": score["status"],
            "cell_hamming": score["cell_hamming"],
        }
        all_results.append(result)

        hamming_str = f"{score['cell_hamming']:.3f}" if score['cell_hamming'] is not None else "N/A"
        print(f"Task {task_index+1}/{len(sampled_ids)} ({task_id}) "
              f"attempt {attempt_index+1}/{NUM_ATTEMPTS}: "
              f"{score['status']}, hamming={hamming_str}")

    # Save after each task for resilience against interruption
    with open(RESULTS_FILE, "w") as f:
        json.dump(all_results, f, indent=2)

print(f"\nDone. {len(all_results)} attempts saved to {RESULTS_FILE}")

### Empty-response handlingOne task (`e6de6e8f`) exhausted the 16,000-token budget on all three attempts without producing any output. Mark these as `budget_exhausted` rather than `unparseable` so the failure mode is distinguishable in analysis.

In [ ]:
with open(RESULTS_FILE) as f:
    results = json.load(f)

for r in results:
    if r["status"] == "unparseable" and r["response_text"] == "":
        r["status"] = "empty_response"
        if r["output_tokens"] >= 16000:
            r["status"] = "budget_exhausted"

with open(RESULTS_FILE, "w") as f:
    json.dump(results, f, indent=2)

budget_exhausted = [r for r in results if r["status"] == "budget_exhausted"]
print(f"Marked {len(budget_exhausted)} attempts as budget_exhausted")

## Generation results: summary statisticsThese cells compute the descriptive statistics referenced in the writeup.

In [ ]:
# Per-task summary
with open(RESULTS_FILE) as f:
    results = json.load(f)

by_task = defaultdict(list)
for r in results:
    by_task[r["task_id"]].append(r)

for task_id, attempts in by_task.items():
    statuses = [a["status"] for a in attempts]
    correct = statuses.count("correct")
    incorrect = statuses.count("incorrect")
    other = len(attempts) - correct - incorrect
    has_wrong = (incorrect > 0) or (other > 0)
    flag = "[has wrong attempt]" if has_wrong else ""
    print(f"  {task_id}: {correct} correct, {incorrect} incorrect, {other} other {flag}")

In [ ]:
# Hamming similarity distribution for wrong attempts
wrong_hammings = [
    r["cell_hamming"] for r in results
    if r["status"] == "incorrect" and r["cell_hamming"] is not None
]

print(f"Wrong attempts with Hamming scores: {len(wrong_hammings)}")
print(f"Mean Hamming:   {np.mean(wrong_hammings):.3f}")
print(f"Median:         {np.median(wrong_hammings):.3f}")
print(f"Min:            {min(wrong_hammings):.3f}")
print(f"Max:            {max(wrong_hammings):.3f}")
print()
print("Distribution:")
for threshold in [0.5, 0.7, 0.8, 0.9, 0.95]:
    n = sum(1 for h in wrong_hammings if h >= threshold)
    pct = 100 * n / len(wrong_hammings)
    print(f"  >= {threshold}: {n} ({pct:.0f}%)")

In [ ]:
# Wrong-cell counts (raw, not proportion). Used for the writeup's claim that
# half of wrong attempts were within 7 cells of correct.
wrong_attempts_detail = []
for r in results:
    if r["status"] == "incorrect" and r["cell_hamming"] is not None:
        gt = r["ground_truth"]
        total_cells = sum(len(row) for row in gt)
        wrong_cells = round(total_cells * (1 - r["cell_hamming"]))
        wrong_attempts_detail.append({
            "task_id": r["task_id"],
            "attempt": r["attempt"],
            "total_cells": total_cells,
            "wrong_cells": wrong_cells,
            "hamming": r["cell_hamming"],
        })

wrong_cell_counts = [w["wrong_cells"] for w in wrong_attempts_detail]
total_cell_counts = [w["total_cells"] for w in wrong_attempts_detail]

print(f"Wrong attempts analyzed: {len(wrong_attempts_detail)}")
print(f"\nWrong cells per attempt:")
print(f"  Mean:   {np.mean(wrong_cell_counts):.1f}")
print(f"  Median: {np.median(wrong_cell_counts):.1f}")
print(f"  Min:    {min(wrong_cell_counts)}")
print(f"  Max:    {max(wrong_cell_counts)}")

print(f"\nGrid size for these attempts:")
print(f"  Mean:   {np.mean(total_cell_counts):.1f}")
print(f"  Median: {np.median(total_cell_counts):.1f}")

print("\nDistribution of wrong cells:")
for threshold in [1, 2, 5, 7, 10, 20]:
    n = sum(1 for w in wrong_cell_counts if w <= threshold)
    pct = 100 * n / len(wrong_cell_counts)
    print(f"  <= {threshold} wrong cells: {n} ({pct:.0f}%)")

# Closest near-misses
print("\nTop 10 closest near-misses (fewest wrong cells):")
wrong_attempts_detail.sort(key=lambda w: w["wrong_cells"])
for w in wrong_attempts_detail[:10]:
    print(f"  {w['task_id']} attempt {w['attempt']}: "
          f"{w['wrong_cells']} wrong of {w['total_cells']} cells "
          f"(hamming {w['hamming']:.3f})")

### Verification: bb52a14b consistencyThe writeup claims that task `bb52a14b` produced an identical wrong output across all three attempts. Verify.

In [ ]:
task_attempts = [r for r in results if r["task_id"] == "bb52a14b"]
print(f"Found {len(task_attempts)} attempts for bb52a14b")

answers = [a["parsed_answer"] for a in task_attempts]
all_same = all(answers[0] == a for a in answers[1:])
print(f"All three attempts identical: {all_same}")

### Verification: color palette analysisThe writeup notes that model wrong attempts preserve the ground truth's color palette 88% of the time but preserve color counts only 19% of the time. This shapes the choice of distractor strategy.

In [ ]:
same_palette_count = 0
same_counts_count = 0
total_checked = 0

for r in results:
    if r["status"] == "incorrect" and r["cell_hamming"] is not None and r["parsed_answer"] is not None:
        gt_flat = [v for row in r["ground_truth"] for v in row]
        ans_flat = [v for row in r["parsed_answer"] for v in row]

        gt_palette = set(gt_flat)
        ans_palette = set(ans_flat)
        gt_counts = Counter(gt_flat)
        ans_counts = Counter(ans_flat)

        if gt_palette == ans_palette:
            same_palette_count += 1
        if gt_counts == ans_counts:
            same_counts_count += 1
        total_checked += 1

print(f"Total wrong attempts checked: {total_checked}")
print(f"Same palette as ground truth: {same_palette_count} ({100*same_palette_count/total_checked:.0f}%)")
print(f"Same color counts as ground truth: {same_counts_count} ({100*same_counts_count/total_checked:.0f}%)")

## Distractor preparationFor each of the 30 tasks, build a distractor for the recognition phase:- For tasks where the model produced at least one parseable wrong attempt: use the wrong attempt with the highest Hamming similarity to ground truth.- For tasks where all three attempts were correct: generate a perturbation distractor by flipping ~5% of cells in the ground truth (preserving the original color palette).Save distractors to `data/distractors.json`.

In [ ]:
DISTRACTORS_FILE = PROJECT_ROOT / "data" / "distractors.json"

with open(RESULTS_FILE) as f:
    results = json.load(f)

# Group attempts by task
by_task = defaultdict(list)
for r in results:
    by_task[r["task_id"]].append(r)

distractors = {}

for task_index, (task_id, attempts) in enumerate(by_task.items()):
    ground_truth = attempts[0]["ground_truth"]

    wrongs = [
        a for a in attempts
        if a["status"] == "incorrect"
        and a["parsed_answer"] is not None
        and a["cell_hamming"] is not None
    ]

    if wrongs:
        # Use the wrong attempt with highest Hamming similarity
        wrongs.sort(key=lambda a: a["cell_hamming"], reverse=True)
        chosen = wrongs[0]
        distractors[task_id] = {
            "distractor_grid": chosen["parsed_answer"],
            "ground_truth": ground_truth,
            "source": "model_attempt",
            "source_attempt_index": chosen["attempt"],
            "hamming_to_truth": chosen["cell_hamming"],
        }
    else:
        # Generate a perturbation distractor
        perturbed = perturb_grid(ground_truth, fraction_to_flip=0.05, seed=task_index)
        total = sum(len(row) for row in ground_truth)
        matching = sum(
            1
            for r1, r2 in zip(perturbed, ground_truth)
            for c1, c2 in zip(r1, r2)
            if c1 == c2
        )
        distractors[task_id] = {
            "distractor_grid": perturbed,
            "ground_truth": ground_truth,
            "source": "perturbation",
            "source_attempt_index": None,
            "hamming_to_truth": matching / total,
        }

with open(DISTRACTORS_FILE, "w") as f:
    json.dump(distractors, f, indent=2)

print(f"Built {len(distractors)} distractors")
model_attempt_count = sum(1 for d in distractors.values() if d["source"] == "model_attempt")
perturbation_count = sum(1 for d in distractors.values() if d["source"] == "perturbation")
print(f"Model attempt distractors: {model_attempt_count}")
print(f"Perturbation distractors: {perturbation_count}")

### Verification: distractor difficultyConfirm the distractor groups have comparable Hamming similarity to ground truth.

In [ ]:
model_hammings = [d["hamming_to_truth"] for d in distractors.values() if d["source"] == "model_attempt"]
perturbation_hammings = [d["hamming_to_truth"] for d in distractors.values() if d["source"] == "perturbation"]

print(f"Model attempt distractors (n={len(model_hammings)}):")
print(f"  Mean Hamming:   {np.mean(model_hammings):.3f}")
print(f"  Median:         {np.median(model_hammings):.3f}")
print()
print(f"Perturbation distractors (n={len(perturbation_hammings)}):")
print(f"  Mean Hamming:   {np.mean(perturbation_hammings):.3f}")
print(f"  Median:         {np.median(perturbation_hammings):.3f}")

## Recognition phaseFor each of the 30 tasks, run two recognition trials with positions counterbalanced:- Trial 1: ground truth in position X, distractor in position Y- Trial 2: distractor in position X, ground truth in position YExtended thinking is disabled. Responses are elicited in `<answer>X</answer>` / `<answer>Y</answer>` format. Save to `data/recognition_results.json`.This phase makes 60 API calls and takes 15-30 minutes. Estimated cost: $3-5.

In [ ]:
RECOGNITION_FILE = PROJECT_ROOT / "data" / "recognition_results.json"

with open(DISTRACTORS_FILE) as f:
    distractors = json.load(f)

recognition_results = []

for task_index, task_id in enumerate(distractors):
    with open(ARC_PATH / f"{task_id}.json") as f:
        task = json.load(f)

    truth = distractors[task_id]["ground_truth"]
    distractor = distractors[task_id]["distractor_grid"]
    distractor_source = distractors[task_id]["source"]

    trials = [
        {"trial": 1, "candidate_x": truth, "candidate_y": distractor, "correct_position": "X"},
        {"trial": 2, "candidate_x": distractor, "candidate_y": truth, "correct_position": "Y"},
    ]

    for trial in trials:
        prompt = build_recognition_prompt(
            task,
            candidate_x=trial["candidate_x"],
            candidate_y=trial["candidate_y"],
        )

        # Recognition: extended thinking disabled
        api_response = call_claude(prompt, thinking_budget=0)

        choice = parse_recognition_response(api_response["text"])
        is_correct = (choice == trial["correct_position"])

        result = {
            "task_id": task_id,
            "trial": trial["trial"],
            "correct_position": trial["correct_position"],
            "model_choice": choice,
            "is_correct": is_correct,
            "distractor_source": distractor_source,
            "model": api_response["model"],
            "timestamp": api_response["timestamp"],
            "input_tokens": api_response["input_tokens"],
            "output_tokens": api_response["output_tokens"],
            "response_text": api_response["text"],
        }
        recognition_results.append(result)

        choice_str = choice if choice else "UNPARSED"
        marker = "Y" if is_correct else ("?" if choice is None else "N")
        print(f"Task {task_index+1}/{len(distractors)} ({task_id}) trial {trial['trial']}: "
              f"correct in {trial['correct_position']}, model chose {choice_str} [{marker}]")

    with open(RECOGNITION_FILE, "w") as f:
        json.dump(recognition_results, f, indent=2)

print(f"\nDone. {len(recognition_results)} trials saved to {RECOGNITION_FILE}")

## Recognition results: summary statistics

In [ ]:
with open(RECOGNITION_FILE) as f:
    recognition_results = json.load(f)

print(f"Total trials: {len(recognition_results)}")

unparsed = [r for r in recognition_results if r["model_choice"] is None]
print(f"Unparsed responses: {len(unparsed)}")

x_choices = sum(1 for r in recognition_results if r["model_choice"] == "X")
y_choices = sum(1 for r in recognition_results if r["model_choice"] == "Y")
print(f"\nModel chose X: {x_choices}")
print(f"Model chose Y: {y_choices}")

correct_trials = sum(1 for r in recognition_results if r["is_correct"])
print(f"\nCorrect trials: {correct_trials}/{len(recognition_results)} = "
      f"{100*correct_trials/len(recognition_results):.1f}%")

# Per-task: did the model get both correct, one, or neither?
rec_by_task = defaultdict(list)
for r in recognition_results:
    rec_by_task[r["task_id"]].append(r)

both_correct = 0
one_correct = 0
neither_correct = 0
for task_id, trials in rec_by_task.items():
    correct_count = sum(1 for t in trials if t["is_correct"])
    if correct_count == 2:
        both_correct += 1
    elif correct_count == 1:
        one_correct += 1
    else:
        neither_correct += 1

print(f"\nTasks correct on both trials: {both_correct}")
print(f"Tasks correct on one trial:   {one_correct}")
print(f"Tasks correct on neither:     {neither_correct}")

# By distractor source
print(f"\nBy distractor source:")
for source in ["model_attempt", "perturbation"]:
    source_trials = [r for r in recognition_results if r["distractor_source"] == source]
    if source_trials:
        correct = sum(1 for r in source_trials if r["is_correct"])
        print(f"  {source}: {correct}/{len(source_trials)} = "
              f"{100*correct/len(source_trials):.1f}%")

## Four-quadrant cross-tabulationThe headline finding: cross-tabulate per-task generation outcome (any of 3 attempts correct) with per-task recognition outcome (both trials correct).

In [ ]:
with open(RESULTS_FILE) as f:
    generation_results = json.load(f)

with open(RECOGNITION_FILE) as f:
    recognition_results = json.load(f)

gen_by_task = defaultdict(list)
for r in generation_results:
    gen_by_task[r["task_id"]].append(r)

rec_by_task = defaultdict(list)
for r in recognition_results:
    rec_by_task[r["task_id"]].append(r)

quadrants = defaultdict(list)
for task_id in gen_by_task:
    gen_correct = sum(1 for a in gen_by_task[task_id] if a["status"] == "correct")
    rec_correct = sum(1 for t in rec_by_task[task_id] if t["is_correct"])

    gen_success = gen_correct > 0
    rec_success = rec_correct == 2

    if gen_success and rec_success:
        quadrants["both_succeed"].append(task_id)
    elif gen_success and not rec_success:
        quadrants["gen_yes_rec_no"].append(task_id)
    elif not gen_success and rec_success:
        quadrants["gen_no_rec_yes"].append(task_id)
    else:
        quadrants["both_fail"].append(task_id)

print(f"Both succeed (gen YES, rec YES):              {len(quadrants['both_succeed'])}")
print(f"Generated but couldn't recognize:             {len(quadrants['gen_yes_rec_no'])}")
print(f"Couldn't generate but recognized:             {len(quadrants['gen_no_rec_yes'])}")
print(f"Both fail (gen NO, rec NO):                   {len(quadrants['both_fail'])}")

print("\nTasks in each quadrant:")
for q, ids in quadrants.items():
    print(f"\n  {q}: {ids}")

### Detail on the gen-yes / rec-no tasksThe four tasks where the model generated correctly at least once but failed at least one recognition trial. Two patterns: 'preferred distractor regardless of position' (real self-evaluation failure) vs. 'picked the same letter both times' (ambiguous at this trial count).

In [ ]:
gen_yes_rec_no_tasks = quadrants["gen_yes_rec_no"]

for task_id in gen_yes_rec_no_tasks:
    trials = [r for r in recognition_results if r["task_id"] == task_id]
    print(f"Task {task_id}:")
    print(f"  Distractor source: {trials[0]['distractor_source']}")
    for t in trials:
        result = "Y" if t["is_correct"] else "N"
        print(f"  Trial {t['trial']}: correct in {t['correct_position']}, "
              f"model chose {t['model_choice']} [{result}]")
    print()

## Chart 1: wrong cells per attemptFor each wrong attempt, show total grid cells (in green) with wrong cells stacked on top (red). Sorted by wrong-cell count, ascending.

In [ ]:
CHARTS_DIR = PROJECT_ROOT / "charts"
CHARTS_DIR.mkdir(exist_ok=True)

with open(RESULTS_FILE) as f:
    results = json.load(f)

data = []
for r in results:
    if r["status"] == "incorrect" and r["cell_hamming"] is not None:
        gt = r["ground_truth"]
        total = sum(len(row) for row in gt)
        wrong = round(total * (1 - r["cell_hamming"]))
        data.append({
            "task_id": r["task_id"],
            "attempt": r["attempt"],
            "total": total,
            "wrong": wrong,
            "correct": total - wrong,
            "label": f"{r['task_id']}.{r['attempt']}",
        })

data.sort(key=lambda d: d["wrong"])

fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(data))
correct_counts = [d["correct"] for d in data]
wrong_counts = [d["wrong"] for d in data]
labels = [d["label"] for d in data]

ax.bar(x, correct_counts, color='#4caf50', edgecolor='gray', alpha=0.5, label='Correct cells')
ax.bar(x, wrong_counts, bottom=correct_counts, color='#e74c3c', edgecolor='darkred', label='Wrong cells')

# Wrong-cell count above each bar
for i, d in enumerate(data):
    ax.text(i, d["total"] + 15, str(d["wrong"]),
            ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=90, fontsize=8)
ax.set_ylabel('Number of cells')
ax.set_xlabel('Wrong attempts (task ID + attempt index, sorted by wrong-cell count)')
ax.set_title("Wrong cells vs. total grid size, per wrong generation attempt\n"
             f"n={len(data)} incorrect attempts, ARC-AGI-1")
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(CHARTS_DIR / 'chart_1_per_attempt.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 2: four-quadrant cross-tabulation

In [ ]:
matrix = np.array([
    [len(quadrants["both_succeed"]),    len(quadrants["gen_yes_rec_no"])],
    [len(quadrants["gen_no_rec_yes"]),  len(quadrants["both_fail"])],
])

fig, ax = plt.subplots(figsize=(8, 6))

ax.imshow(matrix, cmap='Blues', aspect='auto')

for i in range(2):
    for j in range(2):
        count = matrix[i][j]
        text_color = 'white' if count > matrix.max() / 2 else 'black'
        ax.text(j, i, str(count),
                ha='center', va='center',
                fontsize=36, fontweight='bold', color=text_color)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Recognized\n(both trials correct)', 'Did not recognize\n(at least one trial wrong)'], fontsize=11)
ax.set_yticklabels(['Generated\n(at least one of 3 correct)', 'Did not generate\n(no attempts correct)'], fontsize=11)

ax.set_title("Generation vs. Recognition outcomes per task\n"
             "n=30 ARC-AGI-1 tasks, Claude Sonnet 4.6", fontsize=12, pad=15)

ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')

plt.tight_layout()
plt.savefig(CHARTS_DIR / 'chart_2_four_quadrant.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 3: recognition accuracy by distractor source

In [ ]:
sources = ["model_attempt", "perturbation"]
accuracies = []
counts = []

for source in sources:
    source_trials = [r for r in recognition_results if r["distractor_source"] == source]
    correct = sum(1 for r in source_trials if r["is_correct"])
    total = len(source_trials)
    accuracies.append(100 * correct / total)
    counts.append((correct, total))

fig, ax = plt.subplots(figsize=(7, 6))

x = np.arange(len(sources))
labels = ["Model-generated\nnear-miss", "Perturbed\nground truth"]
colors = ['#e74c3c', '#4caf50']

bars = ax.bar(x, accuracies, color=colors, edgecolor='black', width=0.55)

for i, bar in enumerate(bars):
    correct, total = counts[i]
    accuracy = accuracies[i]
    ax.text(bar.get_x() + bar.get_width()/2, accuracy + 1.5,
            f"{accuracy:.1f}%\n({correct}/{total})",
            ha='center', va='bottom', fontsize=11)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('Recognition accuracy (%)')
ax.set_ylim(0, 110)
ax.set_title("Recognition accuracy by distractor type\n"
             "n=60 recognition trials, Claude Sonnet 4.6 on ARC-AGI-1", fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

ax.axhline(50, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Chance (50%)')
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig(CHARTS_DIR / 'chart_3_distractor_source.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 4: worked example for `bb52a14b`Render the test input, the model's wrong output, and the ground truth side by side. The model's three generation attempts produced this same wrong output, missing on 5 cells out of 484. In recognition, the model picked the correct answer in both trials.

In [ ]:
# Standard ARC color palette (matches Chollet's reference implementation)
ARC_COLORS = [
    "#000000",  # 0: black
    "#0074D9",  # 1: blue
    "#FF4136",  # 2: red
    "#2ECC40",  # 3: green
    "#FFDC00",  # 4: yellow
    "#AAAAAA",  # 5: grey
    "#F012BE",  # 6: fuchsia
    "#FF851B",  # 7: orange
    "#7FDBFF",  # 8: teal
    "#870C25",  # 9: brown
]
arc_cmap = ListedColormap(ARC_COLORS)


def render_grid(ax, grid, title=""):
    """Render an ARC grid as a colored square plot on the given axes."""
    arr = np.array(grid)
    ax.imshow(arr, cmap=arc_cmap, vmin=0, vmax=9, interpolation='nearest')
    rows, cols = arr.shape
    for i in range(rows + 1):
        ax.axhline(i - 0.5, color='gray', linewidth=0.3)
    for j in range(cols + 1):
        ax.axvline(j - 0.5, color='gray', linewidth=0.3)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=10)
    ax.set_aspect('equal')


TASK_ID = "bb52a14b"

with open(ARC_PATH / f"{TASK_ID}.json") as f:
    task = json.load(f)

wrong_attempts = [
    r for r in generation_results
    if r["task_id"] == TASK_ID and r["status"] == "incorrect"
]
wrong_grid = wrong_attempts[0]["parsed_answer"]
ground_truth = task["test"][0]["output"]
test_input = task["test"][0]["input"]

total_cells = sum(len(row) for row in ground_truth)
wrong_cells = sum(
    1 for r1, r2 in zip(wrong_grid, ground_truth)
    for c1, c2 in zip(r1, r2) if c1 != c2
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
render_grid(axes[0], test_input, "Test input")
render_grid(axes[1], wrong_grid, f"Claude's output\n({wrong_cells} cells differ from truth)")
render_grid(axes[2], ground_truth, "Ground truth")

fig.suptitle(f"Task {TASK_ID}: Claude produced this same wrong answer 3/3 times.\n"
             "In recognition trials, it correctly identified the ground truth both times.",
             fontsize=12)

plt.tight_layout()
plt.savefig(CHARTS_DIR / 'chart_4_worked_example.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Total cells: {total_cells}")
print(f"Wrong cells: {wrong_cells}")

## End of runAll four charts are saved in `charts/`. The full data is in `data/`. See `WRITEUP.md` for the analysis.